In [435]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [436]:
df_train = pd.read_csv(r'C:\Users\user\Desktop\Skilfactory\data_kaggle\train.csv')
df_test = pd.read_csv(r'C:\Users\user\Desktop\Skilfactory\data_kaggle\test.csv')

In [437]:
def deleate_func(data):
    Passenger = data['PassengerId']
    data.drop(['PassengerId', 'Name', 'Cabin', 'Ticket'], inplace=True, axis=1)
    data['Embarked'] = data['Embarked'].fillna('S')
    return data, Passenger

df_train, _ = deleate_func(df_train)
df_test, passenger = deleate_func(df_test)

In [438]:
def age_predict(data, train='YES'):
    data_one_hot = pd.get_dummies(data)
    data_one_hot['Fare'] = data_one_hot['Fare'].fillna(data_one_hot['Fare'].mean())
    data_Nan = data_one_hot[data_one_hot['Age'].isnull()]
    data_not_Nan = data_one_hot[~data_one_hot['Age'].isnull()]
    if train == 'YES':
        X_train, y_train = data_not_Nan.drop(['Survived', 'Age'], axis=1), data_not_Nan['Age']
        X_test = data_Nan.drop(['Survived', 'Age'], axis=1)
    else:
        X_train, y_train = data_not_Nan.drop(['Age'], axis=1), data_not_Nan['Age']
        X_test = data_Nan.drop(['Age'], axis=1)
        
    lin_reg = LinearRegression()
    lin_reg.fit(X_train, y_train)
    age_predict = lin_reg.predict(X_test).astype('int')
    age_predict_new = np.maximum(age_predict, 20)
    
    age_predict_series = pd.Series(age_predict_new, index=data_Nan.index)
    data_one_hot['Age'] = data_one_hot['Age'].fillna(age_predict_series)
    data_one_hot = data_one_hot.dropna()
    return data_one_hot

df_train = age_predict(df_train)
df_test = age_predict(df_test, train='NO')

In [439]:
df_test

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,3,34.5,0,0,7.8292,False,True,False,True,False
1,3,47.0,1,0,7.0000,True,False,False,False,True
2,2,62.0,0,0,9.6875,False,True,False,True,False
3,3,27.0,0,0,8.6625,False,True,False,False,True
4,3,22.0,1,1,12.2875,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...
413,3,24.0,0,0,8.0500,False,True,False,False,True
414,1,39.0,0,0,108.9000,True,False,True,False,False
415,3,38.5,0,0,7.2500,False,True,False,False,True
416,3,24.0,0,0,8.0500,False,True,False,False,True


In [440]:
X_train, y_train = df_train.drop('Survived', axis=1), df_train['Survived']
X_test = df_test

In [441]:
X_test

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,3,34.5,0,0,7.8292,False,True,False,True,False
1,3,47.0,1,0,7.0000,True,False,False,False,True
2,2,62.0,0,0,9.6875,False,True,False,True,False
3,3,27.0,0,0,8.6625,False,True,False,False,True
4,3,22.0,1,1,12.2875,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...
413,3,24.0,0,0,8.0500,False,True,False,False,True
414,1,39.0,0,0,108.9000,True,False,True,False,False
415,3,38.5,0,0,7.2500,False,True,False,False,True
416,3,24.0,0,0,8.0500,False,True,False,False,True


Тестовая выборка

In [442]:
y_test = pd.read_csv(r'C:\Users\user\Desktop\Skilfactory\data_kaggle\gender_submission.csv')
y_test = y_test.drop('PassengerId', axis=1)

## Логистическая регрессия

In [443]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
params_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced']
}
cv = StratifiedKFold(n_splits=3)
grid = GridSearchCV(estimator=LogisticRegression(random_state=42, max_iter=1000), param_grid=params_grid, scoring='accuracy', cv=cv)
grid.fit(X_train, y_train)
best_log_reg = grid.best_estimator_
y_pred_best_log = best_log_reg.predict(X_test)
print(classification_report(y_test, y_pred_best_log))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94       266
           1       0.89      0.92      0.90       152

    accuracy                           0.93       418
   macro avg       0.92      0.93      0.92       418
weighted avg       0.93      0.93      0.93       418



## Дерево Решений

In [444]:
from sklearn.tree import DecisionTreeClassifier
params_grid = {
    'max_depth': list(range(1, 20)),
    'min_samples_split': [2, 5, 10, 20, 50, 100], 
    'min_samples_leaf': [1, 2, 5, 10, 20, 50], 
    'criterion': ['gini', 'entropy'],
}
cv = StratifiedKFold(n_splits=3)
grid_tree = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=params_grid, scoring='accuracy', cv=cv)
grid_tree.fit(X_train, y_train)
best_tree = grid_tree.best_estimator_
y_pred_best_tree = best_tree.predict(X_test)
print(classification_report(y_test, y_pred_best_tree))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       266
           1       0.96      0.96      0.96       152

    accuracy                           0.97       418
   macro avg       0.97      0.97      0.97       418
weighted avg       0.97      0.97      0.97       418



## Случайный лес

In [445]:
from sklearn.ensemble import RandomForestClassifier
params_grid_rf = {
    'n_estimators': [200, 300, 500],
    'max_depth': [3, 5, 8, 10]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=params_grid_rf,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=3
)

grid_rf.fit(X_train, y_train)
best_forest = grid_rf.best_estimator_
y_pred_best_rf = best_forest.predict(X_test)
print(classification_report(y_test, y_pred_best_rf))

Fitting 3 folds for each of 12 candidates, totalling 36 fits
              precision    recall  f1-score   support

           0       0.88      0.94      0.91       266
           1       0.88      0.78      0.83       152

    accuracy                           0.88       418
   macro avg       0.88      0.86      0.87       418
weighted avg       0.88      0.88      0.88       418



## Adaboost

In [446]:
from sklearn.ensemble import AdaBoostClassifier
params_grid_ada = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.5, 1.0, 1.5, 2.0]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_ada = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=params_grid_ada,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=3
)
grid_ada.fit(X_train, y_train)
best_ada = grid_ada.best_estimator_
y_pred_best_ada = best_ada.predict(X_test)
print(classification_report(y_test, y_pred_best_ada))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
              precision    recall  f1-score   support

           0       0.94      0.92      0.93       266
           1       0.86      0.89      0.88       152

    accuracy                           0.91       418
   macro avg       0.90      0.91      0.90       418
weighted avg       0.91      0.91      0.91       418



In [447]:
from sklearn.ensemble import GradientBoostingClassifier
gr = GradientBoostingClassifier()
gr.fit(X_train, y_train)
y_pred_gr = gr.predict(X_test)
print(classification_report(y_test, y_pred_gr))

              precision    recall  f1-score   support

           0       0.90      0.94      0.92       266
           1       0.89      0.82      0.85       152

    accuracy                           0.89       418
   macro avg       0.89      0.88      0.88       418
weighted avg       0.89      0.89      0.89       418



In [449]:
from sklearn.ensemble import StackingClassifier
estimators = [('tree', best_tree),
              ('ada', best_ada)]

Stack = StackingClassifier(estimators=estimators, 
                           final_estimator=LogisticRegression(),
                           cv=cv)
Stack.fit(X_train, y_train)
y_pred_Stack = Stack.predict(X_test)
print(classification_report(y_test, y_pred_Stack))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       266
           1       0.96      0.96      0.96       152

    accuracy                           0.97       418
   macro avg       0.97      0.97      0.97       418
weighted avg       0.97      0.97      0.97       418



In [457]:
y = pd.Series(y_pred_Stack, index = passenger).to_frame().reset_index()
y = y.rename(columns={0 : 'Survived'})
y

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [458]:
y.to_csv('submission.csv', index=False)

In [452]:
from sklearn.ensemble import VotingClassifier
vote = VotingClassifier(estimators=estimators,
                        voting='hard')
vote.fit(X_train, y_train)
y_pred_vote = vote.predict(X_test)
print(classification_report(y_test, y_pred_vote))

              precision    recall  f1-score   support

           0       0.94      0.98      0.96       266
           1       0.96      0.89      0.92       152

    accuracy                           0.95       418
   macro avg       0.95      0.93      0.94       418
weighted avg       0.95      0.95      0.95       418

